# 02 · Analizar los datos

Trabaja sobre la carpeta de salida de `00` o `01` (por defecto `salida_demo`). Todo lo de aquí sirve
igual para datos reales de SkillCorner, porque el formato es el mismo.

In [ ]:
# 1) INSTALACIÓN — en Colab tarda ~2 min; en tu Mac (ya instalado) no hace nada.
import sys, subprocess, os
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    if not os.path.exists("tracking"):
        # rama con el código; si ya está fusionada en main, git usa la rama por defecto
        r = subprocess.run(["git", "clone", "-q", "-b", "claude/soccernet-calibration-pkg-r8517j", "https://github.com/delioguzmang-maker/tracking"])
        if r.returncode != 0:
            subprocess.run(["git", "clone", "-q", "https://github.com/delioguzmang-maker/tracking"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "tracking[notebooks]"], check=True)
    sys.path.insert(0, os.path.abspath("tracking"))
import soccercal
print("soccercal", soccercal.__version__, "listo")

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from soccercal import pitch
CARPETA = "salida_demo"
if not os.path.exists(f"{CARPETA}/tracking.csv"):
    import soccercal
    soccercal.run(soccercal.get_sample_video(), CARPETA, render=False)
df = pd.read_csv(f"{CARPETA}/tracking.csv")
df.head()

## Dibujar el campo

In [ ]:
def campo(ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(10.5, 6.8))
    ax.set_facecolor("#3a7d44")
    for pl in pitch.polylines():
        if np.abs(pl[:, 2]).max() == 0:
            ax.plot(pl[:, 0], pl[:, 1], color="white", lw=1)
    ax.set_xlim(-57, 57); ax.set_ylim(-38, 38); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    return ax

## Mapa de calor de un jugador (solo posiciones vistas por la cámara)

In [ ]:
jug = df[df.role != "referee"].groupby("player_id").size().idxmax()   # el jugador con más datos
d = df[(df.player_id == jug) & df.is_detected]
ax = campo()
ax.hexbin(d.x, d.y, gridsize=25, extent=(-52.5, 52.5, -34, 34), cmap="hot", alpha=0.7, mincnt=1)
ax.set_title(f"jugador {jug}: {len(d)} posiciones");

## Velocidad a lo largo del tiempo

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
for pid, g in df[df.role == "player"].groupby("player_id"):
    ax.plot(g.timestamp, g.speed_kmh, lw=0.8)
ax.axhline(20, ls="--", c="k", lw=0.8); ax.axhline(25, ls="--", c="r", lw=0.8)
ax.set_xlabel("s"); ax.set_ylabel("km/h"); ax.set_title("velocidad (líneas: 20 km/h HSR, 25 km/h sprint)");

## Forma del equipo: centroide, anchura y profundidad (jugadores vistos)

In [ ]:
vis = df[df.is_detected & (df.role == "player")]
forma = vis.groupby(["frame", "team"]).agg(cx=("x", "mean"), ancho=("y", lambda s: s.max() - s.min()),
                                            largo=("x", lambda s: s.max() - s.min()), n=("x", "size")).reset_index()
forma = forma[forma.n >= 4]
fig, axs = plt.subplots(1, 2, figsize=(12, 3))
for t, g in forma.groupby("team"):
    axs[0].plot(g.frame / 10, g.ancho, label=f"equipo {t}"); axs[1].plot(g.frame / 10, g.largo, label=f"equipo {t}")
axs[0].set_title("anchura (m)"); axs[1].set_title("profundidad (m)"); axs[0].legend();

## Resumen físico

In [ ]:
pd.read_csv(f"{CARPETA}/physical.csv").round(1)